In [1]:
import pandas as pd
from pathlib import Path
RUTA_GOLD = Path("output/gold")


ARCHIVO_CUENTAS = RUTA_GOLD / "master_cuentas_ahorro.parquet"
ARCHIVO_DPF     = RUTA_GOLD / "master_plazo_fijo.parquet"
ARCHIVO_TC      = RUTA_GOLD / "master_tarjetas_credito.parquet"
cuentas_df = pd.read_parquet(ARCHIVO_CUENTAS)
dpf_df = pd.read_parquet(ARCHIVO_DPF)
tc_df = pd.read_parquet(ARCHIVO_TC)

In [7]:
import pandas as pd

# 1. Carga de datos Gold
# tc_df = ... (tu DataFrame de tarjetas)

# 2. Variable del filtro global de banco (cambia 'BCP', 'BBVA', etc., o 'Todos')
banco_seleccionado = "Scotiabank"  # O "Todos"

# 3. Aplicar filtro único por banco
if banco_seleccionado != "Todos":
    tc_filtrado = tc_df[tc_df["banco"] == banco_seleccionado].copy()
else:
    tc_filtrado = tc_df.copy()

# 4. Limpieza básica idéntica a obtener_metricas_tc
tc_filtrado["valor_neto_anual_recurrente_soles"] = pd.to_numeric(
    tc_filtrado["valor_neto_anual_recurrente_soles"], errors="coerce"
)

tc_validas = tc_filtrado.dropna(
    subset=["banco", "valor_neto_anual_recurrente_soles"]
)
tc_validas = tc_validas[
    (tc_validas["banco"].astype(str).str.strip() != "")
    & (tc_validas["valor_neto_anual_recurrente_soles"] > 0)
]

# 5. Obtener el máximo absoluto dentro del filtro de banco
if not tc_validas.empty:
    idx_max = tc_validas["valor_neto_anual_recurrente_soles"].idxmax()
    tarjeta_top = tc_validas.loc[idx_max]

    print("--- RESULTADO VERDADERO ---")
    print(f"Banco: {tarjeta_top['banco']}")
    print(f"Tarjeta: {tarjeta_top['nombre_tarjeta']}")
    print(
        f"Valor Neto Anual: S/ {tarjeta_top['valor_neto_anual_recurrente_soles']:,.2f}"
    )
    print(f"Segmento: {tarjeta_top.get('segmento_gasto', 'N/A')}")
else:
    print("No hay tarjetas válidas para el banco seleccionado.")

--- RESULTADO VERDADERO ---
Banco: Scotiabank
Tarjeta: Visa Infinite
Valor Neto Anual: S/ 1,152.00
Segmento: Elite
